## tl;dr
2026-09-08：三次独立进程复跑，PTO p50 比 native 慢 13.30、15.65、3.34 us。
native device profile 为 6 次 replay × 38 kernel；PTO 当前只有 L1 父 kernel 时间线，逐 core 子任务泳道尚未采到。
本审计只读取本次原始结果，不执行 NPU、不改变 benchmark。

## Context & Methods
A3 device0、TP1、B4/S8、C8191、TRB L1 ACLGraph；每路 20 warmup + 100 sample，ABBA，每 20 enqueue 同步。正式延迟与诊断 profiling 分开。
### Key Assumptions
同一合成 fixture；真实生产算子，但不是 checkpoint/serving 测试，历史 cache 初始多数为零。设备没有硬件队列锁，未宣称跨容器独占。
原始计时是 ns，展示除以 1000 为 us；不删除长尾。图上的原始 device ts 保留，静态预览才减去各自 MODEL_EXECUTE 起点。

## Data
从当前仓内定位本次原始结果与离线审计代码。

In [1]:
from pathlib import Path
import csv, hashlib, json, math, sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'tests/pypto_dsv4_decode_csa/audit_profile_artifacts.py').is_file())
sys.path.insert(0, str(repo))
from tests.pypto_dsv4_decode_csa.audit_profile_artifacts import audit_samples, split_device_traces
root = repo / 'tests/pypto_dsv4_decode_csa/results/20260908_apple_to_apple'
manifest = json.loads((root / 'artifacts/audit_manifest.json').read_text())
for source, expected in manifest['hashes'].items():
    path = Path(source) if Path(source).is_absolute() else repo / source
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, source
print('所有输入文件 SHA256 与采集审计清单一致。')

所有输入文件 SHA256 与采集审计清单一致。


## Results
### 1. 从原始样本重算分位数与 ABBA 位置效应

In [2]:
summaries, samples = audit_samples([root / f'run{i}/result.json' for i in (1, 2, 3)])
assert len(samples) == 600
for row in summaries:
    print(row['run'], row['backend'], 'p50/p90/p99(us)=', *(round(row[k], 3) for k in ('p50_us','p90_us','p99_us')), 'ABBA first/second p50=', row['first_in_abba_p50_us'], row['second_in_abba_p50_us'])
print('保留全部样本；最大值(us)=', max(s['duration_us'] for s in samples))

run1 native_production p50/p90/p99(us)= 720.35 727.89 731.348 ABBA first/second p50= 702.7 725.64
run1 pypto_trb p50/p90/p99(us)= 733.65 744.566 760.227 ABBA first/second p50= 737.15 731.69
run2 native_production p50/p90/p99(us)= 723.07 729.98 734.649 ABBA first/second p50= 705.62 727.65
run2 pypto_trb p50/p90/p99(us)= 738.72 749.582 767.009 ABBA first/second p50= 740.94 736.74
run3 native_production p50/p90/p99(us)= 729.22 736.81 742.008 ABBA first/second p50= 712.51 734.53
run3 pypto_trb p50/p90/p99(us)= 732.56 744.154 752.986 ABBA first/second p50= 737.33 726.89
保留全部样本；最大值(us)= 1244.02


### 2. 检查实际 device 覆盖与输入等价性

In [3]:
trace = next((root / 'profile2/raw').rglob('trace_view.json'))
split = split_device_traces(json.loads(trace.read_text()))
assert all(len(group['windows']) == 6 for group in split.values())
identity = json.loads((root / 'identity/fixture_identity.json').read_text())
assert identity['all_logical_bytes_equal'] and identity['mutable_cache_addresses_disjoint']
assert len(identity['weights_and_buffers']) == 30 and len(identity['initial_caches_and_metadata']) == 21
assert identity['hidden_input']['nonzero_elements'] == 131072
print(json.dumps(manifest['coverage'], indent=2, ensure_ascii=False))
print('30 组权重/buffer、21 组 cache/metadata、hidden input 的逐字节哈希相同。')

{
  "native": {
    "model_id": 49,
    "replays": 6,
    "device_events": 486,
    "kernel_rows": 228,
    "csv_missing_model_ids_resolved_from_device": 6,
    "kernel_types": {
      "DynamicQuant": 24,
      "QuantBatchMatmulV3": 24,
      "RmsNormDynamicQuant": 6,
      "RmsNorm": 6,
      "InplacePartialRotaryMul": 24,
      "ScatterNdUpdateV2": 24,
      "triton_rms_kernel": 6,
      "CompressorMetadata": 12,
      "Compressor": 12,
      "MatMulV2": 24,
      "Muls": 18,
      "Cast": 18,
      "VllmQuantLightningIndexer": 6,
      "SparseAttnSharedkv": 6,
      "TensorMove": 6,
      "Neg": 6,
      "TransposeBatchMatMul": 6
    },
    "detail": "native per-kernel"
  },
  "pypto": {
    "model_id": 48,
    "replays": 6,
    "device_events": 90,
    "kernel_rows": 12,
    "csv_missing_model_ids_resolved_from_device": 12,
    "kernel_types": {
      "simpler_aicpu_l1_exec_9a2ff00fe92aa828": 6,
      "aicore_kernel_0": 6
    },
    "detail": "L1 parent kernels only; no child-task 

### 3. 检查区间并集、重叠与空隙的计算闭合

In [4]:
with (root / 'artifacts/diagnostic_replay_summary.csv').open() as handle:
    replays = list(csv.DictReader(handle))
assert len(replays) == 12
for row in replays:
    assert math.isclose(float(row['kernel_sum_us']) - float(row['kernel_union_us']), float(row['kernel_overlap_us']), abs_tol=1e-6)
    assert math.isclose(float(row['kernel_span_us']) - float(row['kernel_union_us']), float(row['kernel_global_idle_us']), abs_tol=1e-6)
    print(row['backend'], row['replay'], 'kernels=', row['kernel_count'], 'MODEL span(us)=', row['model_span_us'])
print('12 个 replay 区间闭合。注意：PTO overlap 是 AICPU 与父 AICore 的重叠，不是 child-task 并行度。')

native 1 kernels= 38 MODEL span(us)= 751.56
native 2 kernels= 38 MODEL span(us)= 721.2
native 3 kernels= 38 MODEL span(us)= 705.76
native 4 kernels= 38 MODEL span(us)= 728.98
native 5 kernels= 38 MODEL span(us)= 705.9
native 6 kernels= 38 MODEL span(us)= 730.36
pypto 1 kernels= 2 MODEL span(us)= 809.82
pypto 2 kernels= 2 MODEL span(us)= 757.38
pypto 3 kernels= 2 MODEL span(us)= 747.6
pypto 4 kernels= 2 MODEL span(us)= 747.94
pypto 5 kernels= 2 MODEL span(us)= 742.3
pypto 6 kernels= 2 MODEL span(us)= 737.1
12 个 replay 区间闭合。注意：PTO overlap 是 AICPU 与父 AICore 的重叠，不是 child-task 并行度。


## Takeaways
可带限制分享：PTO 在本次合成微基准三轮 p50 均稍慢，不能据此宣称固定慢 17 us 或代表整网吞吐。
native 在 ABBA 的两个位置呈现约 22 us 差别；这是观测到的顺序效应，尚不能确定是 cache、调度或其他原因。
native 逐 kernel device profile 已完整；PTO L1 逐 core 子任务泳道仍需 runtime 诊断采集支持。不能以旧 L2 全零泳道替代。